# Validação 11 — Classificação das relações alegação-evidência

## Goal

Comprovar que cada par `alegação ↔ evidência` pode ser classificado como apoio, contradição, neutralidade ou incerteza, preservando probabilidades, justificativa, modelo e proveniência.

## Setup

Usamos inferência em linguagem natural (NLI) multilíngue com o trecho científico como premissa e a alegação em português como hipótese. O resultado descreve a relação textual encontrada pelo modelo; **não é, isoladamente, um veredito de verdade nem uma avaliação da qualidade do estudo**.

In [1]:
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path
from pprint import pprint
import os
import sys

project_root = Path.cwd()
if not (project_root / 'src').exists():
    project_root = project_root.parent
sys.path.insert(0, str(project_root / 'src'))

from fatofake import (
    DEFAULT_EMBEDDING_MODEL,
    DEFAULT_NLI_MODEL,
    Bm25Index,
    ChunkingConfig,
    ClassificationConfig,
    ExtractionConfig,
    HybridIndex,
    PmcClient,
    PubMedClient,
    RelationLabel,
    SemanticIndex,
    SentenceTransformerEncoder,
    TransformersNliClassifier,
    build_claim_evidence_pairs,
    chunk_article_content,
    classify_claim_evidence_pairs,
    extract_evidence_statements,
    prepare_search_plan,
    retrieve_article_content,
    search_pubmed,
    validate_analysis_input,
)

executed_at = datetime.now(timezone.utc).isoformat()
print(f'Execução UTC: {executed_at}')

Execução UTC: 2026-09-25T11:41:45.229242+00:00


## Steps

### 1. Reexecutar o fluxo até os pares

O mesmo artigo real das validações anteriores é obtido das fontes oficiais, dividido, recuperado e transformado em pares rastreáveis.

In [2]:
class PmidQueryPlanner:
    def generate_queries(self, claim: str) -> list[str]:
        return ['33431520[pmid]']

claim = 'Beber café pode alterar o risco de câncer de próstata.'
analysis_input = validate_analysis_input(claim)
search_plan = prepare_search_plan(analysis_input, PmidQueryPlanner())
pubmed_result = search_pubmed(
    search_plan,
    PubMedClient(email=os.getenv('NCBI_EMAIL'), api_key=os.getenv('NCBI_API_KEY')),
    max_results_per_query=1,
)
content = retrieve_article_content(
    pubmed_result.publications[0],
    PmcClient(email=os.getenv('NCBI_EMAIL'), api_key=os.getenv('NCBI_API_KEY')),
)
chunks = chunk_article_content(content, ChunkingConfig(max_words=120, overlap_words=20))
lexical_index = Bm25Index(chunks)
semantic_index = SemanticIndex(
    chunks,
    SentenceTransformerEncoder(os.getenv('EMBEDDING_MODEL', DEFAULT_EMBEDDING_MODEL)),
)
hybrid_results = HybridIndex(lexical_index, semantic_index).search(claim, top_k=8)
statements = extract_evidence_statements(
    claim,
    hybrid_results,
    ExtractionConfig(max_statements=6, max_per_chunk=2, min_words=8, max_words=80),
)
pairs = build_claim_evidence_pairs(claim, statements)
print(f'Fonte: {content.pmcid} | pares preparados: {len(pairs)}')

/private/tmp/fatofake-notebook-env/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 23444.37it/s]

Fonte: PMC7805365 | pares preparados: 6


### 2. Classificar cada relação com incerteza explícita

Uma classe só é aceita quando a maior probabilidade atinge 60% e supera a segunda colocada por ao menos 10 pontos percentuais. Caso contrário, o resultado apresentado é `UNCERTAIN`, sem descartar as probabilidades originais.

In [3]:
classification_config = ClassificationConfig(
    minimum_confidence=0.60,
    minimum_margin=0.10,
)
classifier = TransformersNliClassifier(os.getenv('NLI_MODEL', DEFAULT_NLI_MODEL))
assessments = classify_claim_evidence_pairs(
    pairs,
    classifier,
    classification_config,
)
distribution = Counter(item.relation.value for item in assessments)
print(f'Modelo: {classifier.name}')
print(f'Distribuição: {dict(distribution)}')

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 17327.06it/s]

Modelo: MoritzLaurer/multilingual-MiniLMv2-L6-mnli-xnli
Distribuição: {'SUPPORTS': 4, 'UNCERTAIN': 1, 'NEUTRAL': 1}


In [4]:
rows = [
    {
        'pair_id': item.pair_id,
        'relation': item.relation.value,
        'model_relation': item.model_relation.value,
        'confidence': round(item.confidence, 4),
        'margin': round(item.margin, 4),
        'p_support': round(item.probabilities.support, 4),
        'p_contradiction': round(item.probabilities.contradiction, 4),
        'p_neutral': round(item.probabilities.neutral, 4),
        'rationale': item.rationale,
        'evidence': item.evidence.text,
        'section': item.evidence.section,
        'source_url': item.evidence.source_url,
    }
    for item in assessments
]
pprint(rows)

[{'confidence': 0.8819,
  'evidence': 'This study suggests that increased coffee consumption may be '
              'associated with a reduced risk of prostate cancer.',
  'margin': 0.7927,
  'model_relation': 'SUPPORTS',
  'p_contradiction': 0.0288,
  'p_neutral': 0.0892,
  'p_support': 0.8819,
  'pair_id': 'pair:f8290b3c652c9f07',
  'rationale': 'O modelo NLI atribuiu maior probabilidade a apoio (88.2%; '
               'margem de 79.3%). Os limites mínimos de confiança e margem '
               'foram atendidos.',
  'relation': 'SUPPORTS',
  'section': 'Conclusions',
  'source_url': 'https://pmc.ncbi.nlm.nih.gov/articles/PMC7805365/'},
 {'confidence': 0.7657,
  'evidence': 'a linear inverse association between coffee consumption and '
              'prostate cancer risk (p=0.006 for linear trend) (figure 3).',
  'margin': 0.5763,
  'model_relation': 'SUPPORTS',
  'p_contradiction': 0.0449,
  'p_neutral': 0.1894,
  'p_support': 0.7657,
  'pair_id': 'pair:3882d00a8e79478c',
  'rationa

## Checks

As verificações garantem que nenhum par foi perdido, todas as probabilidades são válidas, a regra de incerteza foi aplicada e a proveniência continua disponível.

In [5]:
assert len(assessments) == len(pairs)
assert [item.pair_id for item in assessments] == [pair.pair_id for pair in pairs]
assert all(item.model_name == classifier.name for item in assessments)
assert all(item.evidence.pmid == '33431520' for item in assessments)
assert all(item.evidence.pmcid == 'PMC7805365' for item in assessments)
assert all(item.evidence.source_url == content.pmc_url for item in assessments)
assert all(
    abs(
        item.probabilities.support
        + item.probabilities.contradiction
        + item.probabilities.neutral
        - 1.0
    ) < 1e-6
    for item in assessments
)
assert all(0 <= item.confidence <= 1 for item in assessments)
assert all(item.rationale for item in assessments)
assert all(
    item.relation is RelationLabel.UNCERTAIN
    or (
        item.confidence >= classification_config.minimum_confidence
        and item.margin >= classification_config.minimum_margin
    )
    for item in assessments
)
assert all(
    item.relation is not RelationLabel.UNCERTAIN
    or (
        item.confidence < classification_config.minimum_confidence
        or item.margin < classification_config.minimum_margin
    )
    for item in assessments
)

print(
    f'Validação aprovada: {len(assessments)} relações classificadas; '
    'probabilidades, incerteza e proveniência foram preservadas.'
)

Validação aprovada: 6 relações classificadas; probabilidades, incerteza e proveniência foram preservadas.


## Next Steps

A classificação estará validada quando todas as células forem executadas sem erros. A próxima etapa será agregar essas relações por artigo e pela qualidade da evidência, mantendo conflitos visíveis antes de formular uma conclusão ao usuário.